# 📺 MP4 NEWS हिन्दी — Automated TV Broadcast Video Pipeline
### 🚀 Full Offline OmniVoice TTS (High-Performance Multi-Core CPU) + Groq 4-Stage AI + Turbo Remotion Broadcast Renderer

यह नोटबुक Kaggle High-Performance CPU पर संपूर्ण न्यूज़ वीडियो ऑटोमेशन चलाती है:
1. **OmniVoice Neural TTS (k2-fsa):** ऑफ़लाइन मॉडल (`sonuji12345/tts-offline-models`) से बिना वॉइस क्लोन के ओरिजिनल न्यूरल हिंदी आवाज़ (मल्टी-कोर CPU ऑप्टिमाइज़्ड)।
2. **Fast NVMe Staging:** FUSE लेटेंसी से बचने के लिए वज़न सीधे `/tmp/OmniVoice_fast` पर लोड होते हैं (~12s)।
3. **Groq AI 4-स्टेज पाइपलाइन:** वॉइसओवर स्क्रिप्ट (हिंदी संख्याएं), 3-4 डायनामिक हेडलाइंस, असली प्रेस तस्वीरें (0 Memes, 100% Real Photos)।
4. **Turbo Remotion Video Render:** 4-8 पैरेलल वर्कर्स, JPEG फ्रेम पाइपलाइन और VeryFast x264 प्रीसेट के साथ रॉकेट-स्पीड 1080p रेंडरिंग।


---
## ⚡ Step 1: GPU Accelerator Verification & Offline Wheels Setup

In [ ]:
# 1. Environment & Package Setup (High-Performance CPU / GPU Auto-Detection)
import os, sys, subprocess, shutil, glob, time
import torch

cpu_count = os.cpu_count() or 4
print(f"🔥 PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"🚀 Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    torch.set_num_threads(cpu_count)
    print(f"⚡ Active Mode: Kaggle High-Performance CPU ({cpu_count} Threads)")
    try:
        ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
        print(f"💾 High-Capacity System RAM: {ram_gb:.1f} GB (Unlimited Quota!)")
    except Exception:
        pass

# Locate Offline OmniVoice Dataset
OFFLINE_DATASET = ""
for cand in [
    "/kaggle/input/tts-offline-models",
    "/kaggle/input/datasets/sonuji12345/tts-offline-models",
]:
    if os.path.exists(cand):
        OFFLINE_DATASET = cand
        break

if not OFFLINE_DATASET and os.path.exists("/kaggle/input"):
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "OmniVoice" in dirs or "wheels" in dirs:
            OFFLINE_DATASET = root
            break

print("📁 Offline TTS Dataset:", OFFLINE_DATASET or "(Internet download mode)")

# Install offline wheels if available
if OFFLINE_DATASET:
    for root, _, files in os.walk(OFFLINE_DATASET):
        for f in files:
            if f.startswith("omnivoice") and f.endswith(".whl"):
                whl_path = os.path.join(root, f)
                print(f"📦 Installing offline wheel: {whl_path}")
                subprocess.run(["pip", "install", "-q", "--no-deps", whl_path], check=False)

print("📦 Ensuring required Python dependencies (soundfile, groq, beautifulsoup4, feedparser, requests, trafilatura, edge-tts, pillow, googlenewsdecoder)...")
subprocess.run(["pip", "install", "-q", "omnivoice", "soundfile", "groq", "beautifulsoup4", "feedparser", "requests", "trafilatura", "edge-tts", "python-dotenv", "pillow", "googlenewsdecoder", "ddgs", "duckduckgo-search"], check=False)
print("✓ Python environment ready!")

---
## 🎬 Step 2: Node.js & Headless Video Rendering Setup

In [ ]:
# 2. Node.js & Video Renderer Setup
import subprocess, shutil

print('🎬 Setting up Node.js for Remotion rendering...')
if shutil.which('node') is None:
    subprocess.run('curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1', shell=True)
    subprocess.run('apt-get install -y nodejs > /dev/null 2>&1', shell=True)

node_v = subprocess.check_output(['node', '-v'], text=True).strip()
print(f'✓ Node.js version: {node_v}')

# Install headless rendering libraries & Devanagari Hindi fonts for Linux
print('📥 Installing Linux headless graphics & Devanagari fonts (fonts-noto-core, fonts-deva, fonts-indic)...')
subprocess.run('apt-get install -y ffmpeg libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxrandr2 libgbm1 libasound2 fonts-noto-core fonts-deva fonts-indic > /dev/null 2>&1', shell=True)

# Install Google Chrome for 100% pixel-perfect thumbnail and Remotion headless rendering
print('🌐 Ensuring Google Chrome for pixel-perfect thumbnail & broadcast rendering...')
if shutil.which('google-chrome') is None and shutil.which('chromium') is None:
    subprocess.run('wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb && dpkg -i google-chrome-stable_current_amd64.deb > /dev/null 2>&1 || apt-get install -fy -qq > /dev/null 2>&1; rm -f google-chrome-stable_current_amd64.deb', shell=True)

chrome_bin = shutil.which('google-chrome') or shutil.which('chromium') or '/usr/bin/google-chrome'
print(f'✓ Headless render engine & browser ready: {chrome_bin}')


---
## 📂 Step 3: Project Codebase Synchronization & Writable Workspace Setup

In [ ]:
# 3. Project Directory & Fresh GitHub Codebase Synchronization
import os, sys, shutil, subprocess, zipfile

WORK_DIR = "/kaggle/working/Sicreat-automation"
REPO_URL = "https://github.com/9329238475x-wq/Mp4-News-Hindi-Automation.git"

if os.path.exists(WORK_DIR):
    print(f"🧹 Cleaning old working directory {WORK_DIR}...")
    shutil.rmtree(WORK_DIR, ignore_errors=True)

print(f"📦 Cloning latest master codebase from GitHub ({REPO_URL})...")
cloned = False
try:
    res = subprocess.run(["git", "clone", "--depth=1", REPO_URL, WORK_DIR], check=True, timeout=60)
    print("✓ Fresh codebase cloned directly from GitHub!")
    shutil.rmtree(os.path.join(WORK_DIR, ".git"), ignore_errors=True)
    cloned = True
except Exception as e:
    print(f"⚠️ Git clone failed ({e}), checking offline fallback...")

if not cloned or not os.path.exists(os.path.join(WORK_DIR, "niches")):
    SRC_DIR = None
    for cand in [
        "/kaggle/input/sicreat-automation-code",
        "/kaggle/input/datasets/vinodkumartekam/sicreat-automation-code",
    ]:
        if os.path.exists(cand) and os.path.exists(os.path.join(cand, "niches")):
            SRC_DIR = cand
            break
    if SRC_DIR:
        print(f"📦 Copying codebase from {SRC_DIR}...")
        shutil.copytree(SRC_DIR, WORK_DIR, dirs_exist_ok=True)

os.chdir(WORK_DIR)
assert os.path.exists("niches/01-daily-top10/generate_bulletin.py"), "❌ Error: Codebase not initialized!"
print("✓ Verified niches/01-daily-top10/generate_bulletin.py exists!")

# Install Remotion npm packages if node_modules missing
if os.path.exists("package.json") and not os.path.exists("node_modules"):
    print("📦 Installing Remotion npm modules (~30s)...", flush=True)
    subprocess.run("npm install --silent", shell=True)
    print("✓ Remotion modules ready!")

# Register local Hindi TTF fonts in system font cache
subprocess.run('mkdir -p /usr/share/fonts/truetype/noto && cp assets/*.ttf /usr/share/fonts/truetype/noto/ 2>/dev/null; fc-cache -f > /dev/null 2>&1', shell=True)
print('✓ Local Hindi fonts registered in Linux system font cache!')


---
## 🎙️ Step 4: OmniVoice Fast NVMe Staging


In [ ]:
# 4. OmniVoice Fast Staging to NVMe /tmp
import os, sys, time, shutil

# Locate offline weights
MODEL_SOURCE = None
candidates = [
    "/kaggle/input/tts-offline-models/pretrained_models/OmniVoice",
    "/kaggle/input/datasets/sonuji12345/tts-offline-models/pretrained_models/OmniVoice",
    "/kaggle/input/tts-offline-models/OmniVoice",
]
for c in candidates:
    if os.path.exists(c) and (os.path.exists(os.path.join(c, "model.safetensors")) or os.path.exists(os.path.join(c, "config.json"))):
        MODEL_SOURCE = c
        break

if not MODEL_SOURCE and os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "model.safetensors" in files and "tokenizer.json" in files:
            if "omnivoice" in root.lower():
                MODEL_SOURCE = root
                break

if not MODEL_SOURCE:
    MODEL_SOURCE = "k2-fsa/OmniVoice"

# Fast NVMe Staging to /tmp (~10s)
if os.path.isdir(MODEL_SOURCE) and MODEL_SOURCE.startswith("/kaggle/input"):
    local_dir = "/tmp/OmniVoice_fast"
    safetensors_file = os.path.join(local_dir, "model.safetensors")
    if not (os.path.exists(safetensors_file) and os.path.getsize(safetensors_file) > 1024 * 1024):
        print("🚀 Staging OmniVoice neural weights to local NVMe /tmp (~10s)...", flush=True)
        t0 = time.time()
        if os.path.exists(local_dir):
            shutil.rmtree(local_dir, ignore_errors=True)
        shutil.copytree(MODEL_SOURCE, local_dir)
        print(f"✓ Staged in {time.time() - t0:.1f}s!")

    tok_sub = os.path.join(local_dir, "audio_tokenizer")
    if not os.path.isdir(tok_sub) and os.path.exists("/kaggle/input"):
        for root, dirs, _ in os.walk("/kaggle/input"):
            if "audio_tokenizer" in dirs:
                shutil.copytree(os.path.join(root, "audio_tokenizer"), tok_sub)
                break
    os.environ["OMNIVOICE_MODEL_DIR"] = local_dir
    print(f"✓ OmniVoice weights ready at: {local_dir}")
else:
    print(f"✓ OmniVoice model source ready: {MODEL_SOURCE}")


---
## 📰 Step 5: Groq AI 4-Stage News Bulletin Pipeline & Verified Asset Downloader

In [ ]:
# 5. Groq AI News Generation & Master Asset Downloader (100% Fully Automated, Fresh 24-Hour News)
import os, sys, subprocess, json

# Ensure unbuffered real-time stdout streaming in Kaggle logs
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['GROQ_MODEL'] = 'openai/gpt-oss-20b'

def run_command_streaming(cmd, cwd=None, timeout=1200):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in iter(proc.stdout.readline, ''):
        if line:
            print(line, end='', flush=True)
    proc.stdout.close()
    rc = proc.wait(timeout=timeout)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

# Fetch keys from environment, Kaggle user secrets, or obfuscated token pool
env_keys = os.environ.get('GROQ_API_KEYS', '') or os.environ.get('GROQ_API_KEY', '')

if not env_keys:
    try:
        from kaggle_secrets import UserSecretsClient
        env_keys = UserSecretsClient().get_secret('GROQ_API_KEY')
    except Exception:
        pass

if not env_keys:
    _x_tokens = [
        '32263e0a6c252c1c67612d34193b13251f00073f14621f1a0212312c3766130c1d641416202322051b273619071d1b3d020c043632272138',
        '32263e0a1b0f3d07610610261203140f3c2c1810211a32340212312c3766130c141d343960133461333661621f03142020001e3118271632',
        '32263e0a633e05342d3b640d3d16261f1b240f1a1839190c0212312c3766130c1c0561636c1f006116010d67250016382c222c0d1c621f36',
        '32263e0a12623636242500221b321160301367000c25263e0212312c3766130c31160f1f6c60013725050c3a2567660d1417323903620f18',
        '32263e0a140d040c1818102f0c05376d6c2c32100c05113b0212312c3766130c3916613c1e613213140065146c31643a363c220522143b39',
    ]
    env_keys = ','.join(bytes([b ^ 0x55 for b in bytes.fromhex(t)]).decode('utf-8') for t in _x_tokens)

active_keys = [k.strip() for k in env_keys.split(',') if k.strip()]
os.environ['GROQ_API_KEY'] = active_keys[0]
os.environ['GROQ_API_KEYS'] = ','.join(active_keys)

# Determine target story count: checks bulletin_config.json, then env, defaulting to '1' for fast verification
cfg_path = os.path.join(WORK_DIR, 'bulletin_config.json')
target_stories = '1'
if os.path.exists(cfg_path):
    try:
        with open(cfg_path, 'r', encoding='utf-8') as cf:
            c_data = json.load(cf)
            target_stories = str(c_data.get('story_count', '1'))
    except Exception:
        pass
if os.environ.get('STORY_NUMBER'):
    target_stories = os.environ.get('STORY_NUMBER')

print(f'🎯 Broadcast Pipeline Target Stories: {target_stories}', flush=True)

# 100% FULLY AUTOMATED AUTOPILOT: Always fetch the latest breaking news within the last 24 hours
print('📰 [1/3] Fetching 100% fresh, live breaking Hindi news from the last 24 hours (Google News, Dainik Bhaskar, Jagran, BBC, ABP)...', flush=True)
run_command_streaming([sys.executable, 'niches/01-daily-top10/fetch_full_news.py', '--max', target_stories], cwd=WORK_DIR, timeout=1200)

print('🤖 [2/3] Groq AI 4-Stage TV Bulletin Engine: Synthesizing broadcast script, dynamic headlines & search queries...', flush=True)
run_command_streaming([sys.executable, 'niches/01-daily-top10/generate_bulletin.py', '--max', target_stories], cwd=WORK_DIR, timeout=1200)

print('📸 [3/3] Downloading 100% authentic press photos & animated GIFs (Article Media + DuckDuckGo Web Press Photos)...', flush=True)
run_command_streaming([sys.executable, 'niches/01-daily-top10/download_assets.py', '--max', target_stories, '--force'], cwd=WORK_DIR, timeout=1200)
print('✓ Fresh 24-hour news, broadcast script & authentic visual assets ready!', flush=True)



---
## 🔊 Step 6: OmniVoice GPU Audio Generation for News Stories

In [ ]:
# 6. Generate Voiceover Audio using OmniVoice
import os, sys, subprocess

print("🎙️ Synthesizing broadcast Hindi voiceovers with OmniVoice...")
run_command_streaming([sys.executable, "niches/01-daily-top10/generate_audio.py", "--force"], cwd=WORK_DIR, timeout=1800)
print("✓ Voiceover audios generated into out/story_XX/voice over/")


---
## 🎬 Step 7: Remotion TV Broadcast Video Render

In [ ]:
# 7. Turbo Remotion Video Render (Multi-Core CPU Optimized)
import os, time, subprocess, json, glob

out_video = os.path.join(WORK_DIR, 'out', 'Mp4_News_Hindi_Story_01.mp4')
os.makedirs(os.path.dirname(out_video), exist_ok=True)

# Determine whether to render full 50-story bulletin or single story
target_count = 50
cfg_file = os.path.join(WORK_DIR, 'bulletin_config.json')
if os.path.exists(cfg_file):
    try:
        with open(cfg_file, 'r', encoding='utf-8') as cf:
            target_count = int(json.load(cf).get('story_count', 50))
    except Exception:
        pass
if os.environ.get('STORY_NUMBER'):
    try:
        target_count = int(os.environ.get('STORY_NUMBER'))
    except Exception:
        pass

comp_id = 'DLSNewsBulletin' if target_count > 1 else 'DLSNewsSingleStory'

cpu_count = os.cpu_count() or 4
# Kaggle CPU provides 4 cores / 8 vCPUs. Concurrency 4 is stable and high-speed for CPU rendering
render_concurrency = max(2, min(cpu_count, 4))

print(f'🎬 Turbo Rendering MP4 News Hindi broadcast video [{comp_id}] ({render_concurrency} Parallel CPU Workers)...')
t_rend = time.time()
cmd = [
    'npx', 'remotion', 'render',
    comp_id,
    out_video,
    '--overwrite',
    f'--concurrency={render_concurrency}',
    '--image-format=jpeg',
    '--jpeg-quality=85',
    '--x264-preset=veryfast',
    '--gl=angle',
    '--enable-multiprocess-on-linux',
]
print('Command:', ' '.join(cmd))
run_command_streaming(cmd, cwd=WORK_DIR, timeout=3600)
print(f'🎉 Video Render Completed in {time.time() - t_rend:.1f}s!')
print(f'📁 Output Video: {out_video} ({os.path.getsize(out_video)/(1024*1024):.1f} MB)')

print('🖼️ [Auto Thumbnail] Generating Master YouTube Thumbnail (Uncut भारत Template)...', flush=True)
try:
    run_command_streaming([sys.executable, 'niches/01-daily-top10/generate_thumbnail.py'], cwd=WORK_DIR, timeout=300)
    print('✓ Master YouTube Thumbnail Generated successfully!', flush=True)
except Exception as _th_e:
    print(f'⚠️ Thumbnail step notice: {_th_e}', flush=True)



---
## 📺 Step 8: Play Video in Notebook & 1-Click Output Download Zip

In [ ]:
# 8. Play Rendered Video & Prepare Download Zip
import os, shutil, zipfile
from IPython.display import Video, display

out_video = "/kaggle/working/Sicreat-automation/out/Mp4_News_Hindi_Story_01.mp4"
if os.path.exists(out_video):
    size_mb = os.path.getsize(out_video) / (1024 * 1024)
    print(f"✓ Rendered Video: {out_video} ({size_mb:.1f} MB)")
    display(Video(out_video, width=800, embed=True))
    shutil.copy2(out_video, "/kaggle/working/Mp4_News_Hindi_Story_01.mp4")

# Copy thumbnail to root output for direct preview
for t_cand in [
    "/kaggle/working/Sicreat-automation/out/thumbnail_01.jpg",
    "/kaggle/working/Sicreat-automation/out/story_01/thumbnail_01.jpg",
    "/kaggle/working/Sicreat-automation/out/thumbnail.jpg"
]:
    if os.path.exists(t_cand):
        shutil.copy2(t_cand, "/kaggle/working/thumbnail_01.jpg")
        print(f"✓ Master Thumbnail Ready: /kaggle/working/thumbnail_01.jpg")
        break

# Copy full story_01 folder (script, images, audio, data) to root output
story_dir = "/kaggle/working/Sicreat-automation/out/story_01"
if os.path.exists(story_dir):
    shutil.copytree(story_dir, "/kaggle/working/story_01", dirs_exist_ok=True)
    script_f = os.path.join(story_dir, "script_1.txt")
    if os.path.exists(script_f):
        shutil.copy2(script_f, "/kaggle/working/script_1.txt")
    data_f = os.path.join(story_dir, "story_1_data.json")
    if os.path.exists(data_f):
        shutil.copy2(data_f, "/kaggle/working/story_1_data.json")
    print("✓ Story 01 folder, script_1.txt, and metadata exported to output root!")

# Clean up the entire cloned repository folder so Kaggle ONLY returns the output artifacts!
shutil.rmtree("/kaggle/working/Sicreat-automation", ignore_errors=True)
print("✓ Cleaned up working repository, only final artifacts remain!")
